In [ ]:
# =============================================================
# CONFIGURAÇÃO GERAL — Instalações, Imports e Ambiente
# =============================================================

!pip install shap -q

import os
import warnings
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
import shap

warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.0 MB/s eta 0:00:00


✓ SHAP versão: 0.51.0
✓ Todas as bibliotecas importadas com sucesso.
Baixando games.csv...
Downloading...
From: https://drive.google.com/uc?id=1H1SWzyxO3cwCk1W-CJ5CSn1bkNQPUafp
To: /content/games.csv
100% 4.86M/4.86M [00:00<00:00, 51.7MB/s]
Baixando recommendations.csv...
Downloading...
From (original): https://drive.google.com/uc?id=1D6Q_239NtzT3M_A7sD6x1r4XB0_1ypMX
From (redirected): https://drive.google.com/uc?id=1D6Q_239NtzT3M_A7sD6x1r4XB0_1ypMX&confirm=t&uuid=961e06ea-7ab4-4725-8f95-2f08879f4bab
To: /content/recommendations.csv
100% 2.02G/2.02G [00:29<00:00, 69.1MB/s]
Baixando games_metadata.json...
Downloading...
From: https://drive.google.com/uc?id=1no_OD3ljg7FUrVsXjaurjti3DNprxrMK
To: /content/games_metadata.json
100% 17.5M/17.5M [00:00<00:00, 66.0MB/s]


---
## 4. Pipeline de Modelagem Supervisionada

Preparação das features, balanceamento com SMOTE e divisão treino/teste.

In [ ]:
# 1. Criação da variável alvo (Sucesso Comercial = Top 20% faturamento)
df_pipeline = df_games.dropna(subset=['est_revenue_proxy', 'avg_hours', 'price_final', 'positive_ratio']).copy()
threshold_success = df_pipeline['est_revenue_proxy'].quantile(0.80)
df_pipeline['commercial_success'] = (df_pipeline['est_revenue_proxy'] >= threshold_success).astype(int)

# 2. Seleção de atributos para o modelo
# CORREÇÃO: removidos 'user_reviews' e 'price_final' (target leakage)
features_contínuas = ['positive_ratio', 'discount', 'avg_hours']
features_categóricas = ['rating', 'steam_deck']

X_raw = df_pipeline[features_contínuas + features_categóricas].copy()
y = df_pipeline['commercial_success']

# Garantia explícita: colunas de leakage não estão em X
colunas_proibidas = {'user_reviews', 'price_final', 'est_revenue_proxy', 'commercial_success'}
assert colunas_proibidas.isdisjoint(X_raw.columns), "Vazamento de alvo detectado em X!"

# 3. Tratamento de variáveis categóricas (One-Hot Encoding)
X_encoded = pd.get_dummies(X_raw, columns=features_categóricas, drop_first=True)

# Identifica colunas dummy geradas pelo OHE
dummy_cols = [col for col in X_encoded.columns if col not in features_contínuas]

# 4. Divisão dos dados em Treino (80%) e Teste (20%) com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Normalização apenas nas features contínuas (Z-score)
# CORREÇÃO: StandardScaler NÃO aplicado nas dummies de OHE
scaler = StandardScaler()

X_train_cont = pd.DataFrame(
    scaler.fit_transform(X_train[features_contínuas]),
    columns=features_contínuas,
    index=X_train.index  # Mantém índice original
)
X_test_cont = pd.DataFrame(
    scaler.transform(X_test[features_contínuas]),
    columns=features_contínuas,
    index=X_test.index  # Mantém índice original
)

# Recombina: contínuas normalizadas + dummies intactas (0/1)
X_train_ready = pd.concat([X_train_cont, X_train[dummy_cols]], axis=1)
X_test_ready  = pd.concat([X_test_cont,  X_test[dummy_cols]], axis=1)

#Validação de segurança
assert X_train_ready.isna().sum().sum() == 0, "Erro: O DataFrame ainda contém NaNs após a recombinação!"

# 6. Aplicação do SMOTE para balanceamento da base de treino (somente treino)
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_ready, y_train)

# Salvando os conjuntos finais tratados para as próximas etapas
feature_names = X_train_ready.columns.values

np.savez(
    'datasets_modelagem_final.npz',
    X_train=X_train_balanced.values,
    X_test=X_test_ready.values,
    y_train=y_train_balanced.values,
    y_test=y_test.values,
    feature_names=feature_names
)

print("Pipeline de processamento concluído. Dados salvos com sucesso.")
print(f"Features no modelo: {list(feature_names)}")
print(f"Treino balanceado: {X_train_balanced.shape} | Teste: {X_test_ready.shape}")

Pipeline de processamento concluído. Dados salvos com sucesso.
Features no modelo: ['positive_ratio', 'discount', 'avg_hours', 'rating_Mostly Negative', 'rating_Mostly Positive', 'rating_Negative', 'rating_Overwhelmingly Negative', 'rating_Overwhelmingly Positive', 'rating_Positive', 'rating_Very Negative', 'rating_Very Positive', 'steam_deck_True']
Treino balanceado: (65114, 12) | Teste: (10175, 12)


In [ ]:
# Carregamento dos Dados para Modelagem

data = np.load('datasets_modelagem_final.npz', allow_pickle=True)

X_train = data['X_train'].astype(float)
X_test  = data['X_test'].astype(float)
y_train = data['y_train']
y_test  = data['y_test']
feature_names = data['feature_names']

print("Dados carregados com sucesso.")
print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Features: {list(feature_names)}")
print(f"\nDistribuição y_train — 0: {(y_train==0).sum()} | 1: {(y_train==1).sum()}")
print(f"Distribuição y_test  — 0: {(y_test==0).sum()}  | 1: {(y_test==1).sum()}")

Dados carregados com sucesso.
Treino: (65114, 12) | Teste: (10175, 12)
Features: ['positive_ratio', 'discount', 'avg_hours', 'rating_Mostly Negative', 'rating_Mostly Positive', 'rating_Negative', 'rating_Overwhelmingly Negative', 'rating_Overwhelmingly Positive', 'rating_Positive', 'rating_Very Negative', 'rating_Very Positive', 'steam_deck_True']

Distribuição y_train — 0: 32557 | 1: 32557
Distribuição y_test  — 0: 8140  | 1: 2035


---
## 5. Treinamento dos Modelos

Regressão Logística, Random Forest e Gradient Boosting.

In [ ]:
# Treinamento dos Modelos


modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

modelos_treinados = {}
for nome, modelo in modelos.items():
    print(f"Treinando: {nome}...")
    modelo.fit(X_train, y_train)
    modelos_treinados[nome] = modelo
    print(f"  ✓ Concluído")


Treinando: Regressão Logística...
  ✓ Concluído
Treinando: Random Forest...
  ✓ Concluído
Treinando: Gradient Boosting...
  ✓ Concluído


---
## 6. Avaliação dos Modelos Iniciais

In [ ]:
# Avaliação — Métricas Comparativas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

resultados = []

for nome, modelo in modelos_treinados.items():
    y_pred  = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    resultados.append({
        "Modelo":     nome,
        "Acurácia":   round(accuracy_score(y_test, y_pred), 4),
        "Precisão":   round(precision_score(y_test, y_pred), 4),
        "Recall":     round(recall_score(y_test, y_pred), 4),
        "F1-Score":   round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":    round(roc_auc_score(y_test, y_proba), 4),
    })

df_resultados = pd.DataFrame(resultados).set_index("Modelo")
print("=== Tabela Comparativa de Desempenho ===")
display(df_resultados.sort_values("ROC-AUC", ascending=False))

=== Tabela Comparativa de Desempenho ===


,Acurácia,Precisão,Recall,F1-Score,ROC-AUC
Modelo,,,,,
Gradient Boosting,0.7835,0.4758,0.8128,0.6003,0.8819
Regressão Logística,0.7521,0.4326,0.7686,0.5536,0.8370
Random Forest,0.7658,0.4405,0.6329,0.5195,0.8299


In [ ]:
# Curvas ROC — Comparação entre Modelos

fig_roc = go.Figure()

for nome, modelo in modelos_treinados.items():
    y_proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f"{nome} (AUC = {auc:.3f})"
    ))

fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='gray'),
    name='Baseline (Aleatório)'
))

fig_roc.update_layout(
    title="Curvas ROC — Comparação entre Modelos",
    xaxis_title="Taxa de Falsos Positivos (FPR)",
    yaxis_title="Taxa de Verdadeiros Positivos (TPR)",
    template="plotly_white",
    font=dict(size=12, family="Times New Roman"),
    legend=dict(x=0.6, y=0.1)
)
fig_roc.show()

In [ ]:
# Matrizes de Confusão

for nome, modelo in modelos_treinados.items():
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    fig_cm = px.imshow(
        cm,
        text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Predição", y="Real", color="Contagem"),
        x=["Insucesso (0)", "Sucesso (1)"],
        y=["Insucesso (0)", "Sucesso (1)"],
        title=f"Matriz de Confusão — {nome}",
        template="plotly_white"
    )
    fig_cm.update_layout(font=dict(size=12, family="Times New Roman"))
    fig_cm.show()

In [ ]:

# Importância de Features — Melhor Modelo

# Identifica o modelo com maior ROC-AUC na tabela
melhor_nome = df_resultados["ROC-AUC"].idxmax()
melhor_modelo = modelos_treinados[melhor_nome]

print(f"Melhor modelo: {melhor_nome}")

# Extrai importâncias (funciona para RF e GB; para LR usa coeficientes)
if hasattr(melhor_modelo, "feature_importances_"):
    importancias = melhor_modelo.feature_importances_
else:
    importancias = np.abs(melhor_modelo.coef_[0])

df_imp = pd.DataFrame({
    "Feature":    feature_names,
    "Importância": importancias
}).sort_values("Importância", ascending=True)

fig_imp = px.bar(
    df_imp,
    x="Importância",
    y="Feature",
    orientation="h",
    title=f"Importância das Features — {melhor_nome}",
    labels={"Importância": "Importância Relativa", "Feature": "Atributo"},
    template="plotly_white",
    color="Importância",
    color_continuous_scale="Blues"
)
fig_imp.update_layout(
    showlegend=False,
    font=dict(size=12, family="Times New Roman"),
    coloraxis_showscale=False
)
fig_imp.show()

Melhor modelo: Gradient Boosting
